# Ablation A — MLP-LoRA E1 WSFT (Targets FFN Layers)

**Hypothesis:** Adapting only attention layers (q/k/v/o) leaves no capacity for factual storage,
because facts are stored in FFN/MLP layers (Geva et al. 2021, Dai et al. 2022, Meng et al. 2022).
Adding `gate_proj, up_proj, down_proj` and bumping rank to 32 gives the student
dedicated parameters to absorb the teacher's clinical knowledge.

**Why E1 WSFT (not E7):** E1 trains on **all answer tokens** (with explanation upweighted 2x),
giving the student maximum exposure to the teacher's factual content. E7 only supervises
decision tokens (Safe/Unsafe/Caution) and never sees the teacher's explanation facts —
so MLP-LoRA wouldn't have any factual gradient signal to fill the new capacity with.

**Paired comparison:** Original E1 (attention-only LoRA, r=16) is already trained + judged
at all 3 sizes with gemini-3.1. This experiment is **one variable changed**: same loss,
same data, same epochs — only `target_modules` and rank differ.

**Method:** E1 WSFT (Weighted SFT — best balanced Phase 1 method)
**Sizes:** 1.5B + 3B (skip 7B for time)
**Training:** Same 6000 train samples, 2 epochs
**Output:** `runs/e1_mlp_lora/{model}/`

⚠️ Restart kernel between models on Windows.

In [1]:
# Cell 0: Environment + model selection
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["CUDA_DEVICE_MAX_CONNECTIONS"] = "1"

import torch
torch.cuda.set_per_process_memory_fraction(0.85)  # bumped from 0.80

# TRAIN_MODEL = "qwen25_1p5b_base"
#TRAIN_MODEL = "qwen25_3b_base"
TRAIN_MODEL = "qwen25_7b_base"  # ← change to 7B

# 7B needs 8-bit quantization on a 4090
USE_8BIT = "7b" in TRAIN_MODEL
use_bf16 = not USE_8BIT

print(f"GPU: {torch.cuda.get_device_name()}")
print(f"Will train: {TRAIN_MODEL} | 8bit={USE_8BIT} | bf16={use_bf16}")
print("⚠️ Restart kernel before running this")

GPU: NVIDIA GeForce RTX 4090
Will train: qwen25_7b_base | 8bit=True | bf16=False
⚠️ Restart kernel before running this


In [2]:
# Cell 1: Imports + data
import os, sys, json, re
from typing import List, Dict, Any, Tuple
from tqdm import tqdm

import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer,
)
from peft import LoraConfig, get_peft_model

sys.path.insert(0, os.path.expanduser("~/kd_project"))
sys.path.insert(0, r"C:\Users\adishalit1\Desktop\kd_project")
from shared_utils import (
    RUNS_DIR, MAX_SEQ_LEN, LR, SEED,
    W_FORMAT, W_DECISION, W_EXPL,
    STUDENTS, load_data,
    get_section_spans, in_any_span,
    build_prompt_text, tokenize_and_mask, FlexibleCollator,
)

OUT_DIR = os.path.join(RUNS_DIR, "e1_mlp_lora")
os.makedirs(OUT_DIR, exist_ok=True)

prompts, teacher_rows = load_data()
print(f"Output dir: {OUT_DIR}")
print(f"WSFT weights: format={W_FORMAT}, decision={W_DECISION}, explanation={W_EXPL}")

c:\Users\adishalit1\AppData\Local\anaconda3\envs\kd\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Loaded 5000 teacher samples from: data\clinical_pharm_teacher_train_6000.jsonl
Output dir: runs\e1_mlp_lora
WSFT weights: format=1.0, decision=2.0, explanation=2.0


In [3]:
# Cell 2: WSFT Dataset (E1 style — all answer tokens, explanation upweighted)
class WsftDataset(Dataset):
    def __init__(self, rows, prompts, tokenizer, is_instruct):
        print("  Precomputing WSFT dataset...")
        self.items = []
        for r in tqdm(rows, desc="  Tokenizing"):
            prompt = prompts[r["id"]]
            answer = r["teacher_text"]
            prompt_text = build_prompt_text(tokenizer, prompt, is_instruct)
            input_ids, offsets, labels, answer_start = tokenize_and_mask(
                tokenizer, prompt_text, answer
            )

            # Build token-level weights (E1 WSFT style)
            wsft_weights = [0.0] * len(input_ids)
            spans = get_section_spans(answer)
            dec_spans = [(answer_start + s, answer_start + e) for s, e in spans["decision"]]
            expl_spans = [(answer_start + s, answer_start + e) for s, e in spans["explanation"]]

            for i, (s, e) in enumerate(offsets):
                if e <= s:
                    continue
                if s >= answer_start:
                    w = W_FORMAT  # default 1.0 for general answer tokens
                    if in_any_span(s, e, dec_spans):
                        w = W_DECISION  # 2.0
                    if in_any_span(s, e, expl_spans):
                        w = W_EXPL  # 2.0
                    wsft_weights[i] = float(w)

            # Per-sample normalization: mean weight over active tokens = 1
            active_w = [w for w in wsft_weights if w > 0]
            if active_w:
                mw = sum(active_w) / len(active_w)
                if mw > 1e-6:
                    wsft_weights = [w / mw if w > 0 else 0.0 for w in wsft_weights]

            self.items.append({
                "input_ids": torch.tensor(input_ids, dtype=torch.long),
                "attention_mask": torch.ones(len(input_ids), dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long),
                "wsft_weights": torch.tensor(wsft_weights, dtype=torch.float),
            })
        print(f"  ✅ {len(self.items)} samples")

    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]


class WsftTrainer(Trainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._ce = torch.nn.CrossEntropyLoss(reduction="none")

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        wsft_w = inputs.pop("wsft_weights")

        outputs = model(**inputs)
        sl = outputs.logits[:, :-1, :].contiguous()
        ll = labels[:, 1:].contiguous()
        sw = wsft_w[:, 1:].contiguous()
        V = sl.size(-1)

        tok_loss = self._ce(sl.view(-1, V), ll.view(-1)).view(ll.size())
        active = (ll != -100).float()
        w = sw * active
        denom = active.sum(dim=1).clamp(min=1.0)
        loss = ((tok_loss * w).sum(dim=1) / denom).mean()

        return (loss, outputs) if return_outputs else loss

print("✅ WSFT dataset + trainer ready")

✅ WSFT dataset + trainer ready


In [4]:
# # Cell 3: Train E1 WSFT with MLP-LoRA
# cfg = STUDENTS[TRAIN_MODEL]
# out_path = os.path.join(OUT_DIR, TRAIN_MODEL)
# os.makedirs(out_path, exist_ok=True)

# if os.path.exists(os.path.join(out_path, "adapter_config.json")):
#     print(f"⏩ {TRAIN_MODEL} MLP-LoRA already trained")
# else:
#     print(f"\nLoading {TRAIN_MODEL}...")
#     tokenizer = AutoTokenizer.from_pretrained(cfg["path"], trust_remote_code=True)
#     if tokenizer.pad_token is None:
#         tokenizer.pad_token = tokenizer.eos_token

#     base_model = AutoModelForCausalLM.from_pretrained(
#         cfg["path"], torch_dtype=torch.bfloat16,
#         device_map="auto", trust_remote_code=True)

#     # ── KEY DIFFERENCE FROM ORIGINAL E1 ──
#     # Original E1: r=16, target_modules = [q,k,v,o] only (attention)
#     # MLP-LoRA E1: r=32, target_modules = [q,k,v,o, gate_proj, up_proj, down_proj]
#     lora_cfg = LoraConfig(
#         r=32,                    # was 16
#         lora_alpha=64,           # was 32
#         target_modules=[
#             "q_proj", "k_proj", "v_proj", "o_proj",   # attention (same as E1)
#             "gate_proj", "up_proj", "down_proj",      # ✨ NEW: MLP/FFN layers
#         ],
#         lora_dropout=0.05,
#         bias="none",
#         task_type="CAUSAL_LM",
#     )
#     model = get_peft_model(base_model, lora_cfg)
#     model.train()

#     trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
#     total = sum(p.numel() for p in model.parameters())
#     print(f"  ✅ Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
#     print(f"  ✅ {trainable/4_358_144:.1f}x more trainable params than original E1 (r=16, attn-only)")
#     print(f"  ✅ Targets: q/k/v/o (attention) + gate/up/down (MLP) ← key change")

#     dataset = WsftDataset(teacher_rows, prompts, tokenizer, cfg["is_instruct"])
#     collator = FlexibleCollator(tokenizer, extra_1d_fields=["wsft_weights"])

#     trainer = WsftTrainer(
#         model=model,
#         args=TrainingArguments(
#             output_dir=out_path,
#             num_train_epochs=2,                    # same as E1
#             per_device_train_batch_size=1,
#             gradient_accumulation_steps=32,
#             learning_rate=LR,                       # same as E1 (2e-4)
#             logging_steps=25,
#             save_strategy="no",
#             bf16=use_bf16,
#             seed=SEED,
#             report_to="none",
#             remove_unused_columns=False,
#             dataloader_num_workers=0,
#             label_smoothing_factor=0.1,
#         ),
#         train_dataset=dataset,
#         data_collator=collator,
#     )

#     trainer.train()
#     model.save_pretrained(out_path)
#     tokenizer.save_pretrained(out_path)
#     print(f"\n✅ Saved → {out_path}")

In [ ]:
# Cell 3: Train E1 WSFT with MLP-LoRA — supports 8-bit for 7B
from transformers import BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

cfg = STUDENTS[TRAIN_MODEL]
out_path = os.path.join(OUT_DIR, TRAIN_MODEL)
os.makedirs(out_path, exist_ok=True)

if os.path.exists(os.path.join(out_path, "adapter_config.json")):
    print(f"⏩ {TRAIN_MODEL} MLP-LoRA already trained")
else:
    print(f"\nLoading {TRAIN_MODEL}...")
    tokenizer = AutoTokenizer.from_pretrained(cfg["path"], trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    if USE_8BIT:
        bnb = BitsAndBytesConfig(load_in_8bit=True)
        base_model = AutoModelForCausalLM.from_pretrained(
            cfg["path"], quantization_config=bnb,
            device_map="auto", trust_remote_code=True)
        base_model = prepare_model_for_kbit_training(
            base_model, use_gradient_checkpointing=True)
    else:
        base_model = AutoModelForCausalLM.from_pretrained(
            cfg["path"], torch_dtype=torch.bfloat16,
            device_map="auto", trust_remote_code=True)

    # ── KEY: same MLP-LoRA config as 1.5B/3B (one-variable change vs E1) ──
    lora_cfg = LoraConfig(
        r=32,
        lora_alpha=64,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",   # attention
            "gate_proj", "up_proj", "down_proj",      # MLP/FFN
        ],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(base_model, lora_cfg)
    model.train()

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"  ✅ Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

    dataset = WsftDataset(teacher_rows, prompts, tokenizer, cfg["is_instruct"])
    collator = FlexibleCollator(tokenizer, extra_1d_fields=["wsft_weights"])

    trainer = WsftTrainer(
        model=model,
        args=TrainingArguments(
            output_dir=out_path,
            num_train_epochs=2,
            per_device_train_batch_size=1,
            gradient_accumulation_steps=32,
            learning_rate=LR,
            logging_steps=25,
            save_strategy="no",
            bf16=False if USE_8BIT else use_bf16,  # 8-bit uses fp16 internally
            fp16=False,  # let bnb handle it
            seed=SEED,
            report_to="none",
            remove_unused_columns=False,
            dataloader_num_workers=0,
            label_smoothing_factor=0.1,
            gradient_checkpointing=False,  # already enabled by prepare_model_for_kbit_training
        ),
        train_dataset=dataset,
        data_collator=collator,
    )

    trainer.train()
    model.save_pretrained(out_path)
    tokenizer.save_pretrained(out_path)
    print(f"\n✅ Saved → {out_path}")


Loading qwen25_7b_base...


Loading weights: 100%|██████████| 339/339 [00:20<00:00, 16.20it/s]


  ✅ Trainable: 80,740,352 / 7,696,356,864 (1.05%)
  Precomputing WSFT dataset...


  Tokenizing: 100%|██████████| 5000/5000 [00:05<00:00, 954.82it/s] 


  ✅ 5000 samples


c:\Users\adishalit1\AppData\Local\anaconda3\envs\kd\Lib\site-packages\torch\_dynamo\eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
c:\Users\adishalit1\AppData\Local\anaconda3\envs\kd\Lib\site-packages\bitsandbytes\autograd\_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss
25,32.063755
50,26.184360
75,24.771433
100,23.448350
